# SQL Avanzado: Análisis de Datos y Window Functions

Bienvenido a este cuaderno dedicado a técnicas avanzadas de SQL aplicadas a Data Science. Aquí exploraremos cómo extraer información compleja utilizando funciones de ventana (Window Functions), análisis de cohortes, embudos de conversión (funnels) y más.

## 18. Window Functions
Las funciones de ventana realizan cálculos en un conjunto de filas de la tabla que están relacionadas de alguna manera con la fila actual.

In [ ]:
import sqlite3
import pandas as pd

# Crear base de datos en memoria para demostración
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE empleados (
    id INTEGER PRIMARY KEY,
    departamento TEXT,
    salario INTEGER
);
''')

datos = [
    (1, 'Ventas', 3000),
    (2, 'Ventas', 4500),
    (3, 'Ventas', 3000),
    (4, 'IT', 6000),
    (5, 'IT', 7500)
]

cursor.executemany('INSERT INTO empleados VALUES (?, ?, ?)', datos)
conn.commit()

query = '''
SELECT 
    departamento,
    salario,
    AVG(salario) OVER(PARTITION BY departamento) as promedio_depto
FROM empleados;
'''

df = pd.read_sql_query(query, conn)
print(df)

## 19. RANK, DENSE_RANK y ROW_NUMBER
Estas funciones de clasificación asignan números a las filas según ciertos criterios dentro de una partición.

In [ ]:
query_rank = '''
SELECT 
    departamento,
    salario,
    ROW_NUMBER() OVER(PARTITION BY departamento ORDER BY salario DESC) as row_num,
    RANK() OVER(PARTITION BY departamento ORDER BY salario DESC) as rank_val,
    DENSE_RANK() OVER(PARTITION BY departamento ORDER BY salario DESC) as dense_rank_val
FROM empleados;
'''

df_rank = pd.read_sql_query(query_rank, conn)
print(df_rank)

## 20. LAG y LEAD
Permiten acceder a datos de filas anteriores o posteriores sin necesidad de realizar un self-join.

In [ ]:
query_lag_lead = '''
SELECT 
    id,
    salario,
    LAG(salario, 1, 0) OVER(ORDER BY id) as salario_anterior,
    LEAD(salario, 1, 0) OVER(ORDER BY id) as salario_siguiente
FROM empleados;
'''

df_ll = pd.read_sql_query(query_lag_lead, conn)
print(df_ll)

## 21. Running Totals (Totales Acumulados)
Muy útil en finanzas y análisis temporal para calcular sumas acumulativas.

In [ ]:
query_running = '''
SELECT 
    id,
    salario,
    SUM(salario) OVER(ORDER BY id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
FROM empleados;
'''

df_rt = pd.read_sql_query(query_running, conn)
print(df_rt)

## 22. Cohort Analysis en SQL
El análisis de cohortes agrupa a los usuarios según una característica común (por ejemplo, el mes de su primera compra) para analizar su retención a lo largo del tiempo.

In [ ]:
# Simulación simple de transacciones para cohortes
cursor.execute('CREATE TABLE transacciones (user_id INT, fecha TEXT, monto REAL);')
cursor.executemany('INSERT INTO transacciones VALUES (?, ?, ?)', [
    (1, '2023-01-15', 100),
    (1, '2023-02-20', 150),
    (2, '2023-01-20', 200),
    (3, '2023-02-10', 50)
])
conn.commit()

query_cohort = '''
WITH primer_mes AS (
    SELECT user_id, MIN(strftime('%Y-%m', fecha)) as cohort_month
    FROM transacciones
    GROUP BY user_id
)
SELECT cohort_month, COUNT(DISTINCT user_id) as total_usuarios
FROM primer_mes
GROUP BY cohort_month;
'''

df_cohort = pd.read_sql_query(query_cohort, conn)
print(df_cohort)

## 23. Funnel Analysis en SQL
El análisis de embudo mide la progresión de los usuarios a través de una serie de pasos secuenciales (ej. Visita -> Registro -> Compra).

In [ ]:
cursor.execute('CREATE TABLE eventos (user_id INT, paso TEXT);')
cursor.executemany('INSERT INTO eventos VALUES (?, ?)', [
    (1, 'visita'), (1, 'registro'), (1, 'compra'),
    (2, 'visita'), (2, 'registro'),
    (3, 'visita')
])
conn.commit()

query_funnel = '''
SELECT 
    SUM(CASE WHEN paso = 'visita' THEN 1 ELSE 0 END) as visitas,
    SUM(CASE WHEN paso = 'registro' THEN 1 ELSE 0 END) as registros,
    SUM(CASE WHEN paso = 'compra' THEN 1 ELSE 0 END) as compras
FROM eventos;
'''

df_funnel = pd.read_sql_query(query_funnel, conn)
print(df_funnel)

## 24. Advanced SQL Challenges
Retos finales combinando CTEs, Window Functions y Agregaciones para resolver problemas complejos de negocio.

In [ ]:
# Reto: Encontrar el empleado con el segundo salario más alto por departamento
query_challenge = '''
WITH ranked_emp AS (
    SELECT 
        departamento,
        salario,
        DENSE_RANK() OVER(PARTITION BY departamento ORDER BY salario DESC) as rnk
    FROM empleados
)
SELECT departamento, salario
FROM ranked_emp
WHERE rnk = 2;
'''

df_challenge = pd.read_sql_query(query_challenge, conn)
print(df_challenge)

conn.close()